# Pseudo-Label Generation


**Goal:** For each 5-second segment of each soundscape, we assign a pseudo-label *only if the predicted bird species has confidence $\geq$ a non-linear dynamic threshold*.  
The threshold is **lower for rare species** and **higher for common ones**, based on their frequency in the original training data.

**Note:** All the cells except the last 2 ones are this team's contribution. We borrowed the last 2 cells and slightly modfied them to suit our needs as our inference logic is slightly different.

In [1]:
import os
import gc
import warnings
import logging
import time
import math
import cv2
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

In [2]:
class CFG:
 
    test_soundscapes = '/kaggle/input/birdclef-2025/train_soundscapes'
    submission_csv = '/kaggle/input/birdclef-2025/sample_submission.csv'
    taxonomy_csv = '/kaggle/input/birdclef-2025/taxonomy.csv'
    model_path = '/kaggle/input/efficienet-b0-train-birdclef-25'  
    
    # Audio parameters
    FS = 16000  
    WINDOW_SIZE = 5  
    
    # Mel spectrogram parameters
    N_FFT = 512
    HOP_LENGTH = 256
    N_MELS = 64
    FMIN = 50
    FMAX = 8000
    TARGET_SHAPE = (256, 256)
    
    model_name = 'efficientnet_b0'
    in_channels = 1
    device = device = "cuda" if torch.cuda.is_available() else "cpu"

    pseudo_label_threshold = 0.93 # for fallback in case dynamic threshold fails
    
    # Inference parameters
    batch_size = 16
    use_tta = False  
    tta_count = 3   
    threshold = 0.5
    
    use_specific_folds = False  # If False, use all found models
    folds = [0, 1]  # Used only if use_specific_folds is True
    
    debug = False
    debug_count = 3

cfg = CFG()

In [3]:
print(f"Using device: {cfg.device}")
print(f"Loading taxonomy data...")
taxonomy_df = pd.read_csv(cfg.taxonomy_csv)
species_ids = taxonomy_df['primary_label'].tolist()
num_classes = len(species_ids)
print(f"Number of classes: {num_classes}")

Using device: cuda
Loading taxonomy data...
Number of classes: 206


In [4]:
# Step 1: Count number of training samples per class
train_df_original = pd.read_csv("/kaggle/input/birdclef-2025/train.csv")
label_counts = train_df_original['primary_label'].value_counts().to_dict()

# Step 2: Set dynamic thresholds (less common → lower threshold)
min_thresh = 0.55
max_thresh = 0.99

max_count = max(label_counts.values())
min_count = min(label_counts.values())

per_class_thresholds = {}
for label, count in label_counts.items():
    commonness = (count - min_count) / (max_count - min_count + 1e-8)  # 0 = rare, 1 = common
    per_class_thresholds[label] = min_thresh + (commonness ** 0.5) * (max_thresh - min_thresh)  # Apply square root

# Sanity check
print("Dynamic threshold example:")
for label in sorted(label_counts, key=label_counts.get)[:5]:  # 5 least frequent
    print(f"{label} (rare): {per_class_thresholds[label]:.3f}")
for label in sorted(label_counts, key=label_counts.get, reverse=True)[:5]:  # 5 most frequent
    print(f"{label} (common): {per_class_thresholds[label]:.3f}")

Dynamic threshold example:
66531 (rare): 0.550
21116 (rare): 0.550
66578 (rare): 0.550
42113 (rare): 0.550
21038 (rare): 0.550
grekis (common): 0.990
compau (common): 0.947
trokin (common): 0.942
roahaw (common): 0.922
banana (common): 0.895


In [5]:
class BirdCLEFModel(nn.Module):
    def __init__(self, cfg, num_classes):
        super().__init__()
        self.cfg = cfg
        
        self.backbone = timm.create_model(
            cfg.model_name,
            pretrained=False,  
            in_chans=cfg.in_channels,
            drop_rate=0.0,    
            drop_path_rate=0.0
        )
        
        if 'efficientnet' in cfg.model_name:
            backbone_out = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        elif 'resnet' in cfg.model_name:
            backbone_out = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        else:
            backbone_out = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0, '')
        
        self.pooling = nn.AdaptiveAvgPool2d(1)
        self.feat_dim = backbone_out
        self.classifier = nn.Linear(backbone_out, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        
        if isinstance(features, dict):
            features = features['features']
            
        if len(features.shape) == 4:
            features = self.pooling(features)
            features = features.view(features.size(0), -1)
        
        logits = self.classifier(features)
        return logits


In [6]:
def audio2melspec(audio_data, cfg):
    """Convert audio data to mel spectrogram"""
    if np.isnan(audio_data).any():
        mean_signal = np.nanmean(audio_data)
        audio_data = np.nan_to_num(audio_data, nan=mean_signal)

    mel_spec = librosa.feature.melspectrogram(
        y=audio_data,
        sr=cfg.FS,
        n_fft=cfg.N_FFT,
        hop_length=cfg.HOP_LENGTH,
        n_mels=cfg.N_MELS,
        fmin=cfg.FMIN,
        fmax=cfg.FMAX,
        power=2.0
    )

    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec_norm = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
    
    return mel_spec_norm

def process_audio_segment(audio_data, cfg):
    """Process audio segment to get mel spectrogram"""
    if len(audio_data) < cfg.FS * cfg.WINDOW_SIZE:
        audio_data = np.pad(audio_data, 
                          (0, cfg.FS * cfg.WINDOW_SIZE - len(audio_data)), 
                          mode='constant')
    
    mel_spec = audio2melspec(audio_data, cfg)
    
    # Resize if needed
    if mel_spec.shape != cfg.TARGET_SHAPE:
        mel_spec = cv2.resize(mel_spec, cfg.TARGET_SHAPE, interpolation=cv2.INTER_LINEAR)
        
    return mel_spec.astype(np.float32)

In [7]:
def find_model_files(cfg):
    """
    Find all .pth model files in the specified model directory
    """
    model_files = []
    
    model_dir = Path(cfg.model_path)
    
    for path in model_dir.glob('**/*.pth'):
        model_files.append(str(path))
    
    return model_files

def load_models(cfg, num_classes):
    """
    Load all found model files and prepare them for ensemble
    """
    models = []
    
    model_files = find_model_files(cfg)
    
    if not model_files:
        print(f"Warning: No model files found under {cfg.model_path}!")
        return models
    
    print(f"Found a total of {len(model_files)} model files.")
    
    if cfg.use_specific_folds:
        filtered_files = []
        for fold in cfg.folds:
            fold_files = [f for f in model_files if f"fold{fold}" in f]
            filtered_files.extend(fold_files)
        model_files = filtered_files
        print(f"Using {len(model_files)} model files for the specified folds ({cfg.folds}).")
    
    for model_path in model_files:
        try:
            print(f"Loading model: {model_path}")
            checkpoint = torch.load(model_path, map_location=torch.device(cfg.device), weights_only=False)
            
            model = BirdCLEFModel(cfg, num_classes)
            model.load_state_dict(checkpoint['model_state_dict'])
            model = model.to(cfg.device)
            model.eval()
            
            models.append(model)
        except Exception as e:
            print(f"Error loading model {model_path}: {e}")
    
    return models

def predict_on_spectrogram(audio_path, models, cfg, species_ids):
    """Process a single audio file and predict species presence for each 5-second segment"""
    predictions = []
    row_ids = []
    soundscape_id = Path(audio_path).stem
    
    try:
        print(f"Processing {soundscape_id}")
        audio_data, _ = librosa.load(audio_path, sr=cfg.FS)
        
        total_segments = int(len(audio_data) / (cfg.FS * cfg.WINDOW_SIZE))
        
        for segment_idx in range(total_segments):
            start_sample = segment_idx * cfg.FS * cfg.WINDOW_SIZE
            end_sample = start_sample + cfg.FS * cfg.WINDOW_SIZE
            segment_audio = audio_data[start_sample:end_sample]
            
            end_time_sec = (segment_idx + 1) * cfg.WINDOW_SIZE
            row_id = f"{soundscape_id}_{end_time_sec}"
            row_ids.append(row_id)

            if cfg.use_tta:
                all_preds = []
                
                for tta_idx in range(cfg.tta_count):
                    mel_spec = process_audio_segment(segment_audio, cfg)
                    mel_spec = apply_tta(mel_spec, tta_idx)

                    mel_spec = torch.tensor(mel_spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                    mel_spec = mel_spec.to(cfg.device)

                    if len(models) == 1:
                        with torch.no_grad():
                            outputs = models[0](mel_spec)
                            probs = torch.sigmoid(outputs).cpu().numpy().squeeze()
                            all_preds.append(probs)
                    else:
                        segment_preds = []
                        for model in models:
                            with torch.no_grad():
                                outputs = model(mel_spec)
                                probs = torch.sigmoid(outputs).cpu().numpy().squeeze()
                                segment_preds.append(probs)
                        
                        avg_preds = np.mean(segment_preds, axis=0)
                        all_preds.append(avg_preds)

                final_preds = np.mean(all_preds, axis=0)
            else:
                mel_spec = process_audio_segment(segment_audio, cfg)
                
                mel_spec = torch.tensor(mel_spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                mel_spec = mel_spec.to(cfg.device)
                
                if len(models) == 1:
                    with torch.no_grad():
                        outputs = models[0](mel_spec)
                        final_preds = torch.sigmoid(outputs).cpu().numpy().squeeze()
                else:
                    segment_preds = []
                    for model in models:
                        with torch.no_grad():
                            outputs = model(mel_spec)
                            probs = torch.sigmoid(outputs).cpu().numpy().squeeze()
                            segment_preds.append(probs)

                    final_preds = np.mean(segment_preds, axis=0)
                    
            predictions.append(final_preds)
            
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
    
    return row_ids, predictions

In [8]:
def run_inference(cfg, models, species_ids):
    """Run inference on all test soundscapes"""
    test_files = list(Path(cfg.test_soundscapes).glob('*.ogg'))
    
    if cfg.debug:
        print(f"Debug mode enabled, using only {cfg.debug_count} files")
        test_files = test_files[:cfg.debug_count]
    
    print(f"Found {len(test_files)} test soundscapes")

    all_row_ids = []
    all_predictions = []

    for audio_path in tqdm(test_files):
        row_ids, predictions = predict_on_spectrogram(str(audio_path), models, cfg, species_ids)
        all_row_ids.extend(row_ids)
        all_predictions.extend(predictions)
    
    return all_row_ids, all_predictions

In [9]:
models = load_models(cfg, num_classes)

Found a total of 5 model files.
Loading model: /kaggle/input/efficienet-b0-train-birdclef-25/model_fold0.pth
Loading model: /kaggle/input/efficienet-b0-train-birdclef-25/model_fold3.pth
Loading model: /kaggle/input/efficienet-b0-train-birdclef-25/model_fold1.pth
Loading model: /kaggle/input/efficienet-b0-train-birdclef-25/model_fold2.pth
Loading model: /kaggle/input/efficienet-b0-train-birdclef-25/model_fold4.pth


In [10]:
from collections import defaultdict

# temp storage for all predictions per class
class_preds = defaultdict(list)  # label → list of (prob, segment_name, mel)

pseudo_mels = {}
total_segments = 0
audio_files = sorted(Path(cfg.test_soundscapes).glob("*.ogg"))

print(f"Processing {len(audio_files)} files...")
with torch.no_grad():
    for audio_path in tqdm(audio_files, desc="Audio files", dynamic_ncols=True):
        audio_name = audio_path.name
        y, sr = librosa.load(audio_path, sr=cfg.FS)
        if sr != cfg.FS:
            y = librosa.resample(y, orig_sr=sr, target_sr=cfg.FS)

        seg_samples = cfg.FS * cfg.WINDOW_SIZE
        n_segments = int(len(y) / seg_samples)
        total_segments += n_segments

        TOP_N = 3  # max per segment (still useful)
        
        for seg_idx in range(n_segments):
            start = seg_idx * seg_samples
            end = start + seg_samples
            segment = y[start:end]
            mel = process_audio_segment(segment, cfg)
            tensor = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(cfg.device)
        
            if len(models) == 1:
                probs = torch.sigmoid(models[0](tensor)).cpu().numpy().squeeze()
            else:
                preds = [torch.sigmoid(m(tensor)).cpu().numpy().squeeze() for m in models]
                probs = np.mean(np.stack(preds, axis=0), axis=0)
        
            top_n_indices = np.argsort(probs)[-TOP_N:][::-1]
        
            for idx in top_n_indices:
                prob = float(probs[idx])
                label = species_ids[idx]
                threshold = per_class_thresholds.get(label, cfg.pseudo_label_threshold)

                if prob >= threshold:
                    segment_name = f"{audio_name.replace('.ogg', '')}_{seg_idx * cfg.WINDOW_SIZE}.ogg"
                    base = segment_name.replace('.ogg', '')
                    class_preds[label].append((prob, segment_name, mel))


Processing 9726 files...


Audio files:   0%|          | 0/9726 [00:00<?, ?it/s]

In [11]:
# Keep only top-K per class
results = []
results_with_conf = []  # for confidence analysis
MAX_PER_CLASS = 400  # configurable cap

for label, entries in class_preds.items():
    top_entries = sorted(entries, key=lambda x: -x[0])[:MAX_PER_CLASS]
    for prob, segment_name, mel in top_entries:
        results.append([segment_name, label, prob])
        results_with_conf.append({
            "filename": segment_name,
            "primary_label": label,
            "confidence": prob,
            "threshold": per_class_thresholds.get(label, cfg.pseudo_label_threshold)
        })
        pseudo_mels[segment_name.replace('.ogg', '')] = mel

print(f"Done. Collected {len(results)} pseudo-labels from {total_segments} segments.")

# DEBUG: show that our keys line up with the filenames in results
print("First 10 filenames from results (with .ogg):")
print([row[0] for row in results[:10]])
print("First 10 keys in pseudo_mels dict:")
print(list(pseudo_mels.keys())[:10])

print("\nCheck each of the first 10:")
for fn, lbl, prob in results[:10]:
    k = fn.replace(".ogg","")
    print(f"{fn}  → key='{k}'  in dict? {k in pseudo_mels}")

np.save("pseudo_mels.npy", pseudo_mels)

# Save confidence info separately for analysis
conf_df = pd.DataFrame(results_with_conf)
conf_df.to_csv("pseudo_label_confidences.csv", index=False)
print(f"Saved {len(conf_df)} entries with confidence scores to 'pseudo_label_confidences.csv'")

# Save train.csv-compatible format
pseudo_train = pd.DataFrame({
    "filename": [row[0] for row in results],
    "primary_label": [row[1] for row in results],
    "secondary_labels": [[] for _ in results],
    "latitude": [None] * len(results),
    "longitude": [None] * len(results),
    "author": ["pseudo"] * len(results),
    "rating": [0] * len(results),
    "collection": ["pseudo"] * len(results)
})

pseudo_train.to_csv("pseudo_train.csv", index=False)
print(f"Saved {len(pseudo_train)} pseudo-labeled samples to 'pseudo_train.csv'")
pseudo_train.head()


Done. Collected 6279 pseudo-labels from 116712 segments.
First 10 filenames from results (with .ogg):
['H20_20230505_105500_15.ogg', 'H02_20230423_111500_10.ogg', 'H14_20230427_133500_20.ogg', 'H83_20230421_141000_5.ogg', 'H30_20230519_072500_10.ogg', 'H02_20230423_111500_20.ogg', 'H02_20230423_111500_40.ogg', 'H11_20230419_095500_40.ogg', 'H14_20230501_142000_25.ogg', 'H14_20230501_142000_35.ogg']
First 10 keys in pseudo_mels dict:
['H20_20230505_105500_15', 'H02_20230423_111500_10', 'H14_20230427_133500_20', 'H83_20230421_141000_5', 'H30_20230519_072500_10', 'H02_20230423_111500_20', 'H02_20230423_111500_40', 'H11_20230419_095500_40', 'H14_20230501_142000_25', 'H14_20230501_142000_35']

Check each of the first 10:
H20_20230505_105500_15.ogg  → key='H20_20230505_105500_15'  in dict? True
H02_20230423_111500_10.ogg  → key='H02_20230423_111500_10'  in dict? True
H14_20230427_133500_20.ogg  → key='H14_20230427_133500_20'  in dict? True
H83_20230421_141000_5.ogg  → key='H83_20230421_14100

,filename,primary_label,secondary_labels,latitude,longitude,author,rating,collection
0,H20_20230505_105500_15.ogg,blbgra1,[],None,None,pseudo,0,pseudo
1,H02_20230423_111500_10.ogg,blbgra1,[],None,None,pseudo,0,pseudo
2,H14_20230427_133500_20.ogg,blbgra1,[],None,None,pseudo,0,pseudo
3,H83_20230421_141000_5.ogg,blbgra1,[],None,None,pseudo,0,pseudo
4,H30_20230519_072500_10.ogg,blbgra1,[],None,None,pseudo,0,pseudo
